In [1]:
!apt-get update -qq && apt-get install -y -qq openjdk-11-jdk-headless
!pip install -q confluent-kafka fastapi uvicorn nest-asyncio

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package openjdk-11-jre-headless:amd64.
(Reading database ... 118194 files and directories currently installed.)
Preparing to unpack .../openjdk-11-jre-headless_11.0.30+7-1ubuntu1~22.04_amd64.deb ...
Unpacking openjdk-11-jre-headless:amd64 (11.0.30+7-1ubuntu1~22.04) ...
Selecting previously unselected package openjdk-11-jdk-headless:amd64.
Preparing to unpack .../openjdk-11-jdk-headless_11.0.30+7-1ubuntu1~22.04_amd64.deb ...
Unpacking openjdk-11-jdk-headless:amd64 (11.0.30+7-1ubuntu1~22.04) ...
Setting up openjdk-11-jre-headless:amd64 (11.0.30+7-1ubuntu1~22.04) ...
update-alternatives: using /usr/lib/jvm/java-11-openjdk-amd64/bin/jjs to provide /usr/bin/jjs (jjs) in auto mode
update-alternatives: using /usr/lib/jvm/java-11-openjdk-amd64/bin/rmid to provide /usr/bin/rmid

In [ ]:

!fuser -k 8000/tcp 2>/dev/null
!pkill -f "uvicorn" 2>/dev/null
!pkill -f "kafka" 2>/dev/null

In [2]:
import os, urllib.request, tarfile

kafka_version = "3.6.2"
kafka_url = f"https://archive.apache.org/dist/kafka/{kafka_version}/kafka_2.13-{kafka_version}.tgz"
kafka_tgz = "kafka.tgz"

if not os.path.exists("kafka"):
    print("Скачиваем Kafka...")
    urllib.request.urlretrieve(kafka_url, kafka_tgz)
    with tarfile.open(kafka_tgz, "r:gz") as tar:
        tar.extractall()
    os.rename(f"kafka_2.13-{kafka_version}", "kafka")
    print("Готово.")
else:
    print("Kafka уже скачана.")

Скачиваем Kafka...


/tmp/ipykernel_10727/866325428.py:11: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall()


Готово.


In [3]:
import subprocess, time, socket


kafka_dir = "/content/kafka"
zookeeper_props = f"{kafka_dir}/config/zookeeper.properties"
kafka_props = f"{kafka_dir}/config/server.properties"


def port_open(port):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        return s.connect_ex(('localhost', port)) == 0


if not port_open(2181):
    print("Запускаем Zookeeper")
    subprocess.Popen([f"{kafka_dir}/bin/zookeeper-server-start.sh", zookeeper_props],
                     stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    time.sleep(5)
    print("Zookeeper запущен")
else:
    print("Zookeeper уже работает")

if not port_open(9092):
    print("Запускаем Kafka")
    subprocess.Popen([f"{kafka_dir}/bin/kafka-server-start.sh", kafka_props],
                     stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    time.sleep(8)
    print("Kafka запущена.")
else:
    print("Kafka уже работает")

Запускаем Zookeeper
Zookeeper запущен
Запускаем Kafka
Kafka запущена.


In [4]:
def create_topic(topic):
    subprocess.run([f"{kafka_dir}/bin/kafka-topics.sh", "--create",
                    "--bootstrap-server", "localhost:9092",
                    "--replication-factor", "1", "--partitions", "1",
                    "--topic", topic],
                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

for topic in ["transaction.status.requests", "transaction.status.replies"]:
    create_topic(topic)
    print(f"Топик {topic} создан")

Топик transaction.status.requests создан
Топик transaction.status.replies создан


In [5]:
from confluent_kafka import Producer, Consumer, KafkaError
import json, uuid, threading, asyncio
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
import uvicorn, nest_asyncio

# данные

transactions_registry = {
    "tx_001": {"status": "completed", "reason": "clean", "updated": "2025-04-18T10:00:00"},
    "tx_002": {"status": "fraud", "reason": "amount > 150k", "updated": "2025-04-18T10:01:00"},
    "tx_003": {"status": "pending", "reason": "in review", "updated": "2025-04-18T10:02:00"},
    "tx_004": {"status": "completed", "reason": "clean", "updated": "2025-04-18T10:03:00"},
    "tx_005": {"status": "cancelled", "reason": "user rejected", "updated": "2025-04-18T10:04:00"},
    "tx_006": {"status": "fraud", "reason": "velocity > 5 tx/30s", "updated": "2025-04-18T10:05:00"},
    "tx_007": {"status": "completed", "reason": "clean", "updated": "2025-04-18T10:06:00"},
    "tx_008": {"status": "pending", "reason": "manual check", "updated": "2025-04-18T10:07:00"},
    "tx_009": {"status": "completed", "reason": "clean", "updated": "2025-04-18T10:08:00"},
    "tx_010": {"status": "fraud", "reason": "merchant blacklisted", "updated": "2025-04-18T10:09:00"},
    "tx_011": {"status": "completed", "reason": "clean", "updated": "2025-04-18T10:10:00"},
    "tx_012": {"status": "refunded", "reason": "customer dispute", "updated": "2025-04-18T10:11:00"},
    "tx_013": {"status": "completed", "reason": "clean", "updated": "2025-04-18T10:12:00"},
    "tx_014": {"status": "fraud", "reason": "unusual location", "updated": "2025-04-18T10:13:00"},
    "tx_015": {"status": "pending", "reason": "waiting for 3DS", "updated": "2025-04-18T10:14:00"},
}

# Kafka настройки
BROKER = "localhost:9092"
REQ_TOPIC = "transaction.status.requests"
REP_TOPIC = "transaction.status.replies"


reply_producer = Producer({'bootstrap.servers': BROKER})

def delivery_report(err, msg):
    if err:
        print(f"Ошибка доставки ответа: {err}")


def request_consumer_loop():
    consumer = Consumer({
        'bootstrap.servers': BROKER,
        'group.id': 'status_replier',
        'auto.offset.reset': 'earliest',
        'enable.auto.commit': True,
    })
    consumer.subscribe([REQ_TOPIC])
    while True:
        msg = consumer.poll(1.0)
        if msg is None:
            continue
        if msg.error():
            if msg.error().code() == KafkaError._PARTITION_EOF:
                continue
            else:
                print(f"Ошибка consumer: {msg.error()}")
                break
        try:
            value = json.loads(msg.value().decode('utf-8'))
            tx_id = value.get('transaction_id')
            corr_id = value.get('correlation_id')
            if not tx_id or not corr_id:
                continue

            tx_info = transactions_registry.get(tx_id, {"status": "unknown", "reason": "not found"})
            response = {
                "correlation_id": corr_id,
                "transaction_id": tx_id,
                "status": tx_info["status"],
                "details": tx_info["reason"]
            }
            reply_producer.produce(REP_TOPIC, json.dumps(response).encode('utf-8'),
                                   callback=delivery_report)
            reply_producer.flush()
            print(f"Ответ отправлен для {tx_id}, corr_id={corr_id}")
        except Exception as e:
            print(f"Ошибка обработки запроса: {e}")


consumer_thread = threading.Thread(target=request_consumer_loop, daemon=True)
consumer_thread.start()
print("Consumer для запросов запущен в фоне.")

#REST сервис
app = FastAPI()


request_producer = Producer({'bootstrap.servers': BROKER})

# Временное хранилище для ожидающих ответов {correlation_id: Future}
pending_requests = {}
pending_lock = threading.Lock()

def reply_consumer_loop():
    """Слушает топик ответов и заполняет Future для ожидающих запросов"""
    consumer = Consumer({
        'bootstrap.servers': BROKER,
        'group.id': 'rest_reply_receiver',
        'auto.offset.reset': 'earliest',
        'enable.auto.commit': True,
    })
    consumer.subscribe([REP_TOPIC])
    while True:
        msg = consumer.poll(1.0)
        if msg is None:
            continue
        if msg.error():
            continue
        try:
            data = json.loads(msg.value().decode('utf-8'))
            corr_id = data.get('correlation_id')
            with pending_lock:
                if corr_id in pending_requests:
                    future = pending_requests.pop(corr_id)
                    future.set_result(data)
        except Exception:
            pass

reply_consumer_thread = threading.Thread(target=reply_consumer_loop, daemon=True)
reply_consumer_thread.start()

@app.get("/status/{transaction_id}")
async def get_status(transaction_id: str):
    corr_id = str(uuid.uuid4())
    future = asyncio.Future()
    with pending_lock:
        pending_requests[corr_id] = future

    request_msg = {
        "transaction_id": transaction_id,
        "correlation_id": corr_id
    }
    request_producer.produce(REQ_TOPIC, json.dumps(request_msg).encode('utf-8'),
                             callback=lambda err, msg: None)
    request_producer.flush()

    try:
        result = await asyncio.wait_for(future, timeout=5.0)
        return {"transaction_id": result["transaction_id"],
                "status": result["status"],
                "details": result["details"]}   # ← исправлено
    except asyncio.TimeoutError:
        with pending_lock:
            pending_requests.pop(corr_id, None)
        raise HTTPException(status_code=504, detail="Timeout waiting for status")

nest_asyncio.apply()
def run_rest():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="warning")

rest_thread = threading.Thread(target=run_rest, daemon=True)
rest_thread.start()
print("REST сервис запущен на http://localhost:8000")

Consumer для запросов запущен в фоне.
REST сервис запущен на http://localhost:8000


In [6]:
import requests, time

time.sleep(2)

# Проверяем существующие транзакции
for tx_id in ["tx_001", "tx_002", "tx_004", "tx_009"]:
    resp = requests.get(f"http://localhost:8000/status/{tx_id}")
    print(resp.status_code, resp.json())

Ответ отправлен для tx_001, corr_id=dff7290a-9a3e-4089-8d5f-7cd89727be42
200 {'transaction_id': 'tx_001', 'status': 'completed', 'details': 'clean'}
Ответ отправлен для tx_002, corr_id=2d56115d-1482-4805-a932-4281f0a139b7
200 {'transaction_id': 'tx_002', 'status': 'fraud', 'details': 'amount > 150k'}
Ответ отправлен для tx_004, corr_id=b5de5100-7a1b-4e10-8313-0c51bd634abf
200 {'transaction_id': 'tx_004', 'status': 'completed', 'details': 'clean'}
Ответ отправлен для tx_009, corr_id=f97e9252-2923-47f0-a155-fb6ea5b801df
200 {'transaction_id': 'tx_009', 'status': 'completed', 'details': 'clean'}


Как работает request‑reply (синхронный запрос через асинхронную шину)

Клиент (REST‑сервис) отправляет запрос в топик requests и ожидает ответа в топике replies. Чтобы связать ответ с конкретным запросом, используется correlation_id (уникальный для каждого вызова).

Компоненты
REST endpoint (/status/{transaction_id}) – принимает HTTP‑запрос от пользователя.

Request producer – отправляет JSON в transaction.status.requests с полями:

transaction_id – ID проверяемой транзакции.

correlation_id – UUID для идентификации запроса.

Consumer‑обработчик – слушает requests, по transaction_id находит статус (из локального словаря / БД / другого топика), отправляет ответ в replies с тем же correlation_id.

Reply consumer (внутри REST‑сервиса) – слушает replies, по correlation_id передаёт результат ожидающему HTTP‑запросу.

Хранилище pending‑запросов – dict[correlation_id, Future], связывает асинхронные Kafka‑сообщения с ожидающим HTTP‑клиентом.

Поток выполнения (цифры на схеме)
Пользователь → GET /status/tx_001.

REST‑сервис генерирует corr_id = uuid4(), создаёт Future, кладёт в pending_requests[corr_id].

Отправляет сообщение в топик requests: {"transaction_id": "tx_001", "correlation_id": "abc-123"}.

Обработчик (другой consumer) получает сообщение, ищет статус транзакции, отправляет в replies: {"correlation_id": "abc-123", "status": "completed", "details": "clean"}.

Reply consumer внутри REST‑сервиса получает ответ, находит Future по correlation_id, вызывает future.set_result(data).

HTTP‑запрос, ожидавший await future, пробуждается и возвращает ответ клиенту.

Особенности этой реализации
Все компоненты работают в одном процессе (но в реальной системе могут быть распределены).

Таймаут ответа – 5 секунд (если обработчик не ответил, возвращается 504).

Статусы транзакций хранятся в словаре transactions_registry (как БД или кеш из transactions.enriched).